In [1]:
import torch
import transformers
import bert_score

print(f"✅ PyTorch: {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"✅ Transformers: {transformers.__version__}")
print("✅ Все библиотеки загружены успешно!")

c:\Users\Kirill\Desktop\all\ws\pet\MyTextSummarizer\venv_ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ PyTorch: 2.5.1+cu121 (CUDA: True)
✅ Transformers: 5.14.1
✅ Все библиотеки загружены успешно!


In [2]:
import gc
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
from tqdm import tqdm

print(f"Python environment ready")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram:.1f} GB")
else:
    print("⚠️ Working on CPU")

device = "cuda" if torch.cuda.is_available() else "cpu"

NUM_SAMPLES_RU = 200  # Русских примеров
NUM_SAMPLES_EN = 200  # Английских примеров

Python environment ready
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.4 GB


In [3]:
# Пути к моделям
MODELS = {
    "ruT5-base (Baseline)": {
        "path": "cointegrated/rut5-base-absum",
        "prefix": "summarize: ",
        "max_input": 512,
    },
    "mT5-small (Zero-shot)": {
        "path": "google/mt5-small",
        "prefix": "summarize: ",
        "max_input": 512,
    },
    "mT5-small (LoRA Fine-tuned)": {
        "path": "./mt5-small-gazeta-finetuned-lora",
        "prefix": "summarize: ",
        "max_input": 512,
    },
}

def clear_memory():
    """Агрессивная очистка памяти между моделями"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print(f"Memory cleared. Free GPU: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

In [ ]:
def load_model(model_config):
    """Загрузка модели и токенизатора"""
    clear_memory()
    path = model_config["path"]
    print(f"\n📦 Loading {path}...")
    
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSeq2SeqLM.from_pretrained(path, use_safetensors=True)

    model.generation_config.max_length = 150 
    
    model.to(device)
    model.eval()
    
    params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"✅ Loaded ({params:.1f}M parameters) on {device.upper()}")
    return model, tokenizer

In [ ]:
def evaluate_model(model, tokenizer, texts, references, lang='ru', prefix="summarize: ", max_input=512):
    """Генерация + расчет ROUGE и BERTScore с проверкой языка и фиксом для mT5"""

    predictions = []
    print(f"  🔄 Generating {len(texts)} summaries...")
    
    is_mt5 = "mt5" in tokenizer.name_or_path.lower()
    
    with torch.no_grad():
        for text in tqdm(texts, desc=f"  [{lang.upper()}] Generating"):
            inputs = tokenizer(
                prefix + text,
                return_tensors="pt",
                max_length=max_input,
                truncation=True
            ).to(device)
            
            if is_mt5:
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=128,
                    min_length=15,           # Запрещаем слишком короткие ответы
                    do_sample=True,          # Включаем сэмплирование
                    temperature=0.7,         # Контролируем "креативность"
                    top_p=0.9,
                    repetition_penalty=1.0
                )
            else:
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=128,
                    min_length=15,
                    num_beams=4,
                    early_stopping=True,
                    repetition_penalty=1.5,
                    no_repeat_ngram_size=3
                )
                
            pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)
            predictions.append(pred)
    
    # ROUGE
    print(f"  📊 Computing ROUGE...")
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    for ref, pred in zip(references, predictions):
        result = scorer.score(ref, pred)
        for key in rouge_scores:
            rouge_scores[key].append(result[key].fmeasure)
    avg_rouge = {k: sum(v)/len(v) for k, v in rouge_scores.items()}
    
    # BERTScore
    print(f"  📊 Computing BERTScore (lang={lang})...")
    device_idx = 0 if device == "cuda" else -1
    P, R, F1 = bert_score_fn(
        predictions, references, 
        lang=lang, 
        device=device_idx, 
        verbose=False,
        batch_size=8
    )
    
    return {
        "rouge1": avg_rouge['rouge1'],
        "rouge2": avg_rouge['rouge2'],
        "rougeL": avg_rouge['rougeL'],
        "bertscore_f1": F1.mean().item(),
        "predictions": predictions
    }

In [ ]:
print("загрузка датасетов...")

for var in ['texts_ru', 'refs_ru', 'texts_en', 'refs_en']:
    if var in locals():
        del locals()[var]
gc.collect()

# 2. Русский: Gazeta
NUM_SAMPLES_RU = 100
print("Загрузка Russian (Gazeta)...")
dataset_ru = load_dataset("IlyaGusev/gazeta", split=f"test[:{NUM_SAMPLES_RU}]")
texts_ru = [item['text'] for item in dataset_ru]
refs_ru = [item['summary'] for item in dataset_ru]
print(f"✅ Russian: {len(texts_ru)} samples")

загрузка датасетов...
Загрузка Russian (Gazeta)...


✅ Russian: 100 samples
Загрузка English (XSum)...
✅ English: 100 samples
📝 TEXT:
Prison Link Cymru had 1,099 referrals in 2015-16 and said some ex-offenders were living rough for up to a year before finding suitable accommodation.
Workers at the charity claim investment in housing would be cheaper than jailing homeless repeat offenders.
The Welsh Government said more people than...

🎯 REFERENCE:
There is a "chronic" need for more housing for prison leavers in Wales, according to a charity.



In [7]:
print("="*70)
print(f"🇷🇺 EVALUATION ON RUSSIAN (Gazeta, {len(texts_ru)} samples)")
print("="*70)

results_ru = {}
all_predictions_ru = {}

for name, config in MODELS.items():
    print(f"\n{'─'*70}")
    print(f"🔬 Evaluating: {name}")
    print(f"{'─'*70}")
    
    model, tokenizer = load_model(config)
    res = evaluate_model(
        model, tokenizer, texts_ru, refs_ru,
        lang='ru', prefix=config["prefix"], max_input=config["max_input"]
    )
    results_ru[name] = res
    all_predictions_ru[name] = res.pop("predictions")
    
    # Освобождаем память перед следующей моделью
    del model, tokenizer
    clear_memory()

df_ru = pd.DataFrame({
    "Model": list(results_ru.keys()),
    "ROUGE-1": [results_ru[m]['rouge1'] for m in results_ru],
    "ROUGE-2": [results_ru[m]['rouge2'] for m in results_ru],
    "ROUGE-L": [results_ru[m]['rougeL'] for m in results_ru],
    "BERTScore F1": [results_ru[m]['bertscore_f1'] for m in results_ru],
})

# Форматирование
for col in ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BERTScore F1"]:
    df_ru[col] = df_ru[col].map(lambda x: f"{x:.4f}")

print("\n" + "="*70)
print("🇷🇺 RUSSIAN RESULTS")
print("="*70)
print(df_ru.to_markdown(index=False))

🇷🇺 EVALUATION ON RUSSIAN (Gazeta, 100 samples)

──────────────────────────────────────────────────────────────────────
🔬 Evaluating: ruT5-base (Baseline)
──────────────────────────────────────────────────────────────────────
Memory cleared. Free GPU: 5.32 GB

📦 Loading cointegrated/rut5-base-absum...


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 4585.38it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ Loaded (244.3M parameters) on CUDA
✅ Проверка языка пройдена: данные действительно на RU
  🔄 Generating 100 summaries...


  [RU] Generating: 100%|██████████| 100/100 [03:08<00:00,  1.88s/it]


  📊 Computing ROUGE...
  📊 Computing BERTScore (lang=ru)...


...

✅ Loaded (300.2M parameters) on CUDA
✅ Проверка языка пройдена: данные действительно на RU
  🔄 Generating 100 summaries...


  [RU] Generating: 100%|██████████| 100/100 [02:50<00:00,  1.70s/it]


  📊 Computing ROUGE...
  📊 Computing BERTScore (lang=ru)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4873.71it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Memory cleared. Free GPU: 5.28 GB

🇷🇺 RUSSIAN RESULTS
| Model                       |   ROUGE-1 |   ROUGE-2 |   ROUGE-L |   BERTScore F1 |
|:----------------------------|----------:|----------:|----------:|---------------:|
| ruT5-base (Baseline)        |    0.1817 |    0.0543 |    0.1779 |         0.7206 |
| mT5-small (Zero-shot)       |    0.0155 |    0      |    0.0155 |         0.5851 |
| mT5-small (LoRA Fine-tuned) |    0.157  |    0.0572 |    0.153  |         0.6984 |


In [10]:
print("="*70)
print("QUALITATIVE ANALYSIS (3 Russian examples)")
print("="*70)

for i in range(3):
    print(f"\n{'─'*70}")
    print(f"📄 Example {i+1}")
    print(f"{'─'*70}")
    print(f"📝 ORIGINAL:\n{texts_ru[i][:500]}...\n")
    print(f"🎯 REFERENCE:\n{refs_ru[i]}\n")
    for name in MODELS.keys():
        print(f"🤖 {name}:\n{all_predictions_ru[name][i]}\n")

QUALITATIVE ANALYSIS (3 Russian examples)

──────────────────────────────────────────────────────────────────────
📄 Example 1
──────────────────────────────────────────────────────────────────────
📝 ORIGINAL:
На этих выходных в Берлине прошли крупные акции протеста против введенных для борьбы с коронавирусом ограничений. Демонстранты скандировали «Путин!» По словам депутата городской палаты представителей Гуннара Линдеманна («Альтернатива для Германии»), люди выкрикивали фамилию российского президента из уважения к нему. В комментарии РИА «Новости» немецкий политик отметил, что среди населения Германии Владимир Путин имеет хорошую репутацию. По его мнению, протестующие ранее пришли к российскому посо...

🎯 REFERENCE:
Протестующие против антикоронавирусных мер немцы скандировали имя российского президента, потому что уважают его. Такое мнение выразил депутат городской палаты представителей Гуннар Линдеманн. На этих выходных в Берлине прошли крупные акции протеста. Манифестанты требовали

Хотя дообученная модель mT5-small (LoRA) показывает метрики (ROUGE-1, BERTScore) незначительно ниже, чем узкоспециализированный бейзлайн ruT5-base, она является более предпочтительным выбором для продакшена по следующим причинам:
- **Экономичность поддержки:** Использование LoRA позволяет адаптировать модель под новые доменные данные или исправлять ошибки, обучая лишь ~0.16% параметров. Это снижает стоимость и время переобучения на порядки по сравнению с полным файнтюнингом.
- **Отказоустойчивость к шумным данным:** Оригинальная mT5-small была обучена работать со 101 языком. После переобучения под русскоязычные новости она не утратила эти умения, что позволяет ей обрабатывать статьи с несколькими языками, а также потенциал для дообучения под нужные языки
- **Эффективный деплой:** Компактный размер архитектуры (~300M параметров) делает модель идеальным кандидатом для пост-тренировочного квантования (например, до 4-бит). Это позволяет развернуть инференс на дешевых CPU-серверах.
**Итог:** Незначительное снижение метрик (в пределах 2-3%) с лихвой компенсируется гибкостью, дешевизной эксплуатации и устойчивостью модели в реальных условиях.